In [0]:
%sql
DESCRIBE TABLE proyecto_final.bronze.devoluciones;

In [0]:
%sql
SELECT *
FROM proyecto_final.bronze.devoluciones
LIMIT 20;

In [0]:
%sql
SELECT
    COUNT(*) AS total_bronze,
    COUNT(DISTINCT devolucion_id) AS devoluciones_distintas,
    COUNT(*) - COUNT(DISTINCT devolucion_id) AS duplicados_adicionales,
    SUM(
        CASE
            WHEN venta_id IS NULL OR TRIM(venta_id) = '' THEN 1
            ELSE 0
        END
    ) AS ventas_id_vacios,
    SUM(
        CASE
            WHEN motivo IS NULL OR TRIM(motivo) = '' THEN 1
            ELSE 0
        END
    ) AS motivos_vacios
FROM proyecto_final.bronze.devoluciones;

In [0]:
%sql
WITH base AS (
    SELECT
        d.*,
        TRY_CAST(
            REPLACE(TRIM(d.monto_reembolso), ',', '.')
            AS DECIMAL(12,2)
        ) AS monto_convertido,
        COALESCE(
            TRY_CAST(d.fecha_devolucion AS DATE),
            TO_DATE(d.fecha_devolucion, 'dd/MM/yyyy')
        ) AS fecha_convertida
    FROM proyecto_final.bronze.devoluciones d
),
analisis AS (
    SELECT
        b.*,
        v.venta_id AS venta_encontrada,
        v.monto_total AS monto_venta,
        v.sucursal_id AS sucursal_venta,
        v.producto_id AS producto_venta,
        v.fecha_venta
    FROM base b
    LEFT JOIN proyecto_final.silver.ventas v
        ON b.venta_id = v.venta_id
)
SELECT
    COUNT(*) AS total_bronze,

    SUM(CASE
        WHEN monto_convertido IS NULL THEN 1 ELSE 0
    END) AS montos_invalidos,

    SUM(CASE
        WHEN monto_convertido < 0 THEN 1 ELSE 0
    END) AS reembolsos_negativos,

    SUM(CASE
        WHEN monto_convertido > monto_venta THEN 1 ELSE 0
    END) AS reembolsos_superiores_venta,

    SUM(CASE
        WHEN fecha_convertida IS NULL THEN 1 ELSE 0
    END) AS fechas_invalidas,

    SUM(CASE
        WHEN venta_encontrada IS NULL THEN 1 ELSE 0
    END) AS ventas_no_encontradas,

    SUM(CASE
        WHEN venta_encontrada IS NOT NULL
             AND sucursal_id <> sucursal_venta
        THEN 1 ELSE 0
    END) AS sucursales_diferentes,

    SUM(CASE
        WHEN venta_encontrada IS NOT NULL
             AND producto_id <> producto_venta
        THEN 1 ELSE 0
    END) AS productos_diferentes,

    SUM(CASE
        WHEN venta_encontrada IS NOT NULL
             AND fecha_convertida < CAST(fecha_venta AS DATE)
        THEN 1 ELSE 0
    END) AS devoluciones_antes_venta

FROM analisis;

In [0]:
%sql
WITH base AS (
    SELECT
        d.*,
        TRY_CAST(
            REPLACE(TRIM(d.monto_reembolso), ',', '.')
            AS DECIMAL(12,2)
        ) AS monto_convertido,
        COALESCE(
            TRY_CAST(d.fecha_devolucion AS DATE),
            TRY_TO_DATE(d.fecha_devolucion, 'dd/MM/yyyy')
        ) AS fecha_convertida
    FROM proyecto_final.bronze.devoluciones d
),
deduplicadas AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY devolucion_id
            ORDER BY fecha_ingesta DESC, archivo_origen DESC
        ) AS rn
    FROM base
),
evaluadas AS (
    SELECT
        d.*,
        v.venta_id AS venta_encontrada,
        v.monto_total AS monto_venta,
        CASE
            WHEN v.venta_id IS NULL THEN 1 ELSE 0
        END AS venta_huerfana,
        CASE
            WHEN monto_convertido < 0 THEN 1 ELSE 0
        END AS reembolso_negativo,
        CASE
            WHEN monto_convertido > v.monto_total THEN 1 ELSE 0
        END AS reembolso_excesivo,
        CASE
            WHEN fecha_convertida IS NULL THEN 1 ELSE 0
        END AS fecha_invalida
    FROM deduplicadas d
    LEFT JOIN proyecto_final.silver.ventas v
        ON d.venta_id = v.venta_id
    WHERE d.rn = 1
)
SELECT
    COUNT(*) AS devoluciones_unicas,
    SUM(venta_huerfana) AS ventas_huerfanas,
    SUM(reembolso_negativo) AS reembolsos_negativos,
    SUM(reembolso_excesivo) AS reembolsos_excesivos,
    SUM(fecha_invalida) AS fechas_invalidas,
    SUM(
        CASE
            WHEN venta_huerfana = 1
              OR reembolso_negativo = 1
              OR reembolso_excesivo = 1
              OR fecha_invalida = 1
            THEN 1 ELSE 0
        END
    ) AS total_rechazos_unicos,
    SUM(
        CASE
            WHEN venta_huerfana = 0
             AND reembolso_negativo = 0
             AND reembolso_excesivo = 0
             AND fecha_invalida = 0
            THEN 1 ELSE 0
        END
    ) AS total_aceptadas
FROM evaluadas;

In [0]:
%sql
WITH duplicados AS (
    SELECT
        devolucion_id,
        COUNT(*) AS apariciones,
        COUNT(DISTINCT TO_JSON(NAMED_STRUCT(
            'venta_id', venta_id,
            'sucursal_id', sucursal_id,
            'producto_id', producto_id,
            'motivo', motivo,
            'monto_reembolso', monto_reembolso,
            'fecha_devolucion', fecha_devolucion
        ))) AS versiones_distintas
    FROM proyecto_final.bronze.devoluciones
    GROUP BY devolucion_id
),
base AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY devolucion_id
            ORDER BY fecha_ingesta DESC, archivo_origen DESC
        ) AS rn
    FROM proyecto_final.bronze.devoluciones
),
reembolsos AS (
    SELECT
        b.venta_id,
        SUM(
            TRY_CAST(
                REPLACE(TRIM(b.monto_reembolso), ',', '.')
                AS DECIMAL(12,2)
            )
        ) AS total_reembolsado,
        MAX(v.monto_total) AS monto_venta
    FROM base b
    INNER JOIN proyecto_final.silver.ventas v
        ON b.venta_id = v.venta_id
    WHERE b.rn = 1
      AND TRY_CAST(
          REPLACE(TRIM(b.monto_reembolso), ',', '.')
          AS DECIMAL(12,2)
      ) >= 0
    GROUP BY b.venta_id
)
SELECT
    (SELECT COUNT(*)
     FROM duplicados
     WHERE apariciones > 1) AS ids_duplicados,

    (SELECT SUM(apariciones - 1)
     FROM duplicados
     WHERE apariciones > 1) AS filas_duplicadas,

    (SELECT COUNT(*)
     FROM duplicados
     WHERE versiones_distintas > 1) AS duplicados_con_datos_diferentes,

    (SELECT COUNT(*)
     FROM reembolsos
     WHERE total_reembolsado > monto_venta)
       AS ventas_con_reembolso_acumulado_excesivo;

In [0]:
%sql
WITH deduplicadas AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY devolucion_id
            ORDER BY fecha_ingesta DESC, archivo_origen DESC
        ) AS rn
    FROM proyecto_final.bronze.devoluciones
),
enriquecidas AS (
    SELECT
        d.devolucion_id,
        d.venta_id,
        TRY_CAST(
            REPLACE(TRIM(d.monto_reembolso), ',', '.')
            AS DECIMAL(12,2)
        ) AS monto_reembolso,
        v.monto_total AS monto_venta
    FROM deduplicadas d
    INNER JOIN proyecto_final.silver.ventas v
        ON d.venta_id = v.venta_id
    WHERE d.rn = 1
),
validas_individualmente AS (
    SELECT *
    FROM enriquecidas
    WHERE monto_reembolso >= 0
      AND monto_reembolso <= monto_venta
),
acumulados AS (
    SELECT
        venta_id,
        COUNT(*) AS cantidad_devoluciones,
        SUM(monto_reembolso) AS total_reembolsado,
        MAX(monto_venta) AS monto_venta
    FROM validas_individualmente
    GROUP BY venta_id
)
SELECT
    COUNT(*) AS ventas_con_devoluciones_validas,
    SUM(
        CASE WHEN cantidad_devoluciones > 1
             THEN 1 ELSE 0 END
    ) AS ventas_con_multiples_devoluciones,
    SUM(
        CASE WHEN total_reembolsado > monto_venta
             THEN 1 ELSE 0 END
    ) AS ventas_con_exceso_acumulado_valido
FROM acumulados;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Leer las tablas
df_bronze_dev = spark.table(
    "proyecto_final.bronze.devoluciones"
)

df_ventas_ref = (
    spark.table("proyecto_final.silver.ventas")
    .select(
        F.col("venta_id").alias("venta_ref"),
        F.col("sucursal_id").alias("sucursal_ref"),
        F.col("producto_id").alias("producto_ref"),
        F.col("monto_total").alias("monto_venta"),
        F.col("fecha_venta").alias("fecha_venta_ref")
    )
)

# 2. Normalizar montos y fechas
df_dev_normalizadas = (
    df_bronze_dev
    .withColumn(
        "monto_decimal",
        F.expr("""
            TRY_CAST(
                REPLACE(TRIM(monto_reembolso), ',', '.')
                AS DECIMAL(12,2)
            )
        """)
    )
    .withColumn(
        "fecha_date",
        F.coalesce(
            F.expr("TRY_CAST(fecha_devolucion AS DATE)"),
            F.to_date(
                F.expr(
                    "try_to_timestamp(fecha_devolucion, 'dd/MM/yyyy')"
                )
            )
        )
    )
)

# 3. Deduplicar por devolucion_id
ventana = (
    Window.partitionBy("devolucion_id")
    .orderBy(
        F.col("fecha_ingesta").desc_nulls_last(),
        F.col("archivo_origen").desc_nulls_last()
    )
)

df_dev_deduplicadas = (
    df_dev_normalizadas
    .withColumn("rn", F.row_number().over(ventana))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 4. Enriquecer con Silver Ventas
df_dev_enriquecidas = (
    df_dev_deduplicadas
    .join(
        df_ventas_ref,
        F.col("venta_id") == F.col("venta_ref"),
        "left"
    )
)

# 5. Evaluar reglas y conservar todos los motivos
df_calidad_dev = (
    df_dev_enriquecidas
    .withColumn(
        "motivo_rechazo",
        F.concat_ws(
            ", ",
            F.when(
                F.col("devolucion_id").isNull()
                | (F.trim("devolucion_id") == ""),
                "DEVOLUCION_ID_INVALIDO"
            ),
            F.when(
                F.col("venta_ref").isNull(),
                "VENTA_HUERFANA"
            ),
            F.when(
                F.col("monto_decimal").isNull()
                | (F.col("monto_decimal") < 0),
                "REEMBOLSO_INVALIDO"
            ),
            F.when(
                F.col("monto_decimal") > F.col("monto_venta"),
                "REEMBOLSO_EXCESIVO"
            ),
            F.when(
                F.col("fecha_date").isNull(),
                "FECHA_INVALIDA"
            ),
            F.when(
                F.col("sucursal_id").isNull()
                | (F.trim("sucursal_id") == "")
                | (
                    F.col("venta_ref").isNotNull()
                    & ~F.col("sucursal_id").eqNullSafe(
                        F.col("sucursal_ref")
                    )
                ),
                "SUCURSAL_INVALIDA"
            ),
            F.when(
                F.col("producto_id").isNull()
                | (F.trim("producto_id") == "")
                | (
                    F.col("venta_ref").isNotNull()
                    & ~F.col("producto_id").eqNullSafe(
                        F.col("producto_ref")
                    )
                ),
                "PRODUCTO_INVALIDO"
            ),
            F.when(
                F.col("fecha_date").isNotNull()
                & F.col("fecha_venta_ref").isNotNull()
                & (
                    F.col("fecha_date")
                    < F.to_date("fecha_venta_ref")
                ),
                "FECHA_ANTERIOR_VENTA"
            )
        )
    )
    .withColumn(
        "rechazado",
        F.length("motivo_rechazo") > 0
    )
)

# 6. Preparar las dos salidas sin escribir tablas
df_silver_dev = (
    df_calidad_dev
    .filter(~F.col("rechazado"))
    .select(
        "devolucion_id",
        "venta_id",
        "sucursal_id",
        "producto_id",
        "motivo",
        F.col("monto_decimal").alias("monto_reembolso"),
        F.col("fecha_date").alias("fecha_devolucion")
    )
)

df_rechazos_dev = df_calidad_dev.filter(
    F.col("rechazado")
)

# 7. Validación antes de escribir
print("Bronze:", df_bronze_dev.count())
print("Sin duplicados:", df_dev_deduplicadas.count())
print("Silver preparada:", df_silver_dev.count())
print("Cuarentena preparada:", df_rechazos_dev.count())

display(
    df_rechazos_dev
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
TABLA_SILVER_DEV = "proyecto_final.silver.devoluciones"

# Verificar antes de escribir
assert df_silver_dev.count() == 3892
assert df_rechazos_dev.count() == 265

# Guardar Silver
(
    df_silver_dev.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER_DEV)
)

# Comprobar la carga
df_verificacion = spark.table(TABLA_SILVER_DEV)

print("Registros cargados:", df_verificacion.count())
print(
    "Devoluciones distintas:",
    df_verificacion.select("devolucion_id").distinct().count()
)

assert df_verificacion.count() == 3892
assert df_verificacion.select("devolucion_id").distinct().count() == 3892

display(df_verificacion.limit(20))

In [0]:
%sql
SELECT
    devolucion_id,
    venta_id,
    sucursal_id,
    producto_id,
    motivo,
    monto_reembolso,
    fecha_devolucion
FROM proyecto_final.silver.devoluciones
ORDER BY devolucion_id
LIMIT 30;

In [0]:
from pyspark.sql import functions as F

columnas_bronze_dev = df_bronze_dev.columns

df_cuarentena_dev = (
    df_rechazos_dev
    .select(
        F.lit("devoluciones").alias("fuente"),
        F.col("devolucion_id").alias("clave_registro"),
        F.col("motivo_rechazo"),
        F.current_timestamp().alias("fecha_rechazo"),
        F.to_json(
            F.struct(
                *[F.col(c) for c in columnas_bronze_dev]
            ),
            options={"ignoreNullFields": "false"}
        ).alias("datos_originales"),
        F.col("archivo_origen")
    )
)

print("Registros preparados:", df_cuarentena_dev.count())

print(
    "Devoluciones distintas:",
    df_cuarentena_dev.select("clave_registro").distinct().count()
)

print(
    "JSON originales nulos:",
    df_cuarentena_dev.filter(
        F.col("datos_originales").isNull()
    ).count()
)

assert df_cuarentena_dev.count() == 265

display(
    df_cuarentena_dev
    .groupBy("motivo_rechazo")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
from pyspark.sql import functions as F

TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"

# Verificar datos preparados
assert df_cuarentena_dev.count() == 265

# Consultar registros existentes por fuente
print("ANTES DE LA CARGA")
display(
    spark.table(TABLA_CUARENTENA)
    .groupBy("fuente")
    .count()
    .orderBy("fuente")
)

# Eliminar únicamente devoluciones de ejecuciones anteriores
spark.sql(f"""
    DELETE FROM {TABLA_CUARENTENA}
    WHERE fuente = 'devoluciones'
""")

# Insertar los rechazos de Devoluciones
(
    df_cuarentena_dev
    .select(
        "fuente",
        "clave_registro",
        "motivo_rechazo",
        "fecha_rechazo",
        "datos_originales",
        "archivo_origen"
    )
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(TABLA_CUARENTENA)
)

# Validar carga y distribución de fuentes
print("DESPUÉS DE LA CARGA")
display(
    spark.table(TABLA_CUARENTENA)
    .groupBy("fuente")
    .count()
    .orderBy("fuente")
)

total_dev = (
    spark.table(TABLA_CUARENTENA)
    .filter(F.col("fuente") == "devoluciones")
    .count()
)

assert total_dev == 265

print("Devoluciones en cuarentena:", total_dev)

In [0]:
%sql
SELECT
    fuente,
    clave_registro,
    motivo_rechazo,
    fecha_rechazo,
    datos_originales,
    archivo_origen
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'devoluciones'
ORDER BY clave_registro
LIMIT 30;

In [0]:
%sql
SELECT
    COUNT(*) AS total_cuarentena,
    COUNT(DISTINCT clave_registro) AS devoluciones_distintas,
    SUM(
        CASE WHEN datos_originales IS NULL
             THEN 1 ELSE 0 END
    ) AS json_originales_nulos
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'devoluciones';